In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
def make_map(files, dlon=1, dsine=1e-2, stonyhurst=False, mu_thr=0.1):
    s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
    xd, yd = s['xd'], s['yd']

    df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).dropna()
    dids = df['did'].to_numpy()
    xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

    grid = np.mgrid[-1:1 + dsine / 2:dsine, :360:dlon]
    grid[0] = np.arcsin(grid[0].clip(-1,1)) * 180 / np.pi

    mean, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0])

    for file in files[:]:
        with fits.open(file) as hdul:
            header = hdul[0].header.copy()
            data = hdul[0].data.copy()

        did = int(file.split('_')[-1].split('.')[0])
        xd_, yd_ = crop_grid(xd, yd, header)

        view = View.from_header(header)
        view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

        transform = view.to_synoptic(correct_mu=True, correct_dr=True, mu_thr=mu_thr, stonyhurst=stonyhurst) + ToSpherical()

        grid_, alpha = (~transform)(grid)
        grid_ = (interp2d(xd_, *grid_, kind='bilinear'), interp2d(yd_, *grid_, kind='bilinear'))
        data = interp2d(data, *grid_) * alpha

        n = (~np.isnan(data)) * (1 / alpha) ** 4
        coverage += np.nan_to_num(n)
        mean += np.nan_to_num((data - mean) * n / coverage)

    mean[coverage == 0] = np.nan
    coverage[coverage == 0] = np.nan
    return mean

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = np.array([datetime.fromisoformat(date) for date in df['date']])
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()

np.unique(car_rot)

array([2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289,
       2290, 2291, 2292, 2293, 2294, 2295, 2296, 2297, 2298, 2299, 2300,
       2301, 2302, 2303, 2304, 2305, 2306])

In [4]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*')))


dsine = 0.004
dlon = 0.5

lat = np.arange(-1,1 + dsine / 2,0.004)
lat = np.arcsin(lat.clip(-1,1)) * 180 / np.pi

lon = np.arange(0,360,dlon)

for i in np.unique(car_rot):
    print(i)

    t = np.where(car_rot == i)[0]
    mean = make_map(files[t], dsine=0.004, dlon=dlon)

    np.savez(f'temp/{i}.npz', data=mean, lat=lat, lon=lon, start=dates[t[0]], end=dates[t[-1]])

2279
2280
2281
2282
2283
2284
2285
2286
2287
2288
2289
2290
2291
2292
2293
2294
2295
2296
2297
2298
2299
2300
2301
2302
2303
2304
2305
2306


In [11]:
plt.figure(figsize=(14,8))
plt.imshow(mean, aspect='auto', origin='lower', vmin=-1000, vmax=1000, cmap='hmimag', interpolation='bicubic')
plt.tight_layout()

In [6]:
files = sorted(glob.glob('temp/*'))

Q = []
for file in files:
    s = np.load(file)
    data = s['data']
    Q += [np.nanmean(data, axis=1)]

Q = np.array(Q)

/tmp/ipykernel_36519/497070043.py:7: RuntimeWarning: Mean of empty slice
  Q += [np.nanmean(data, axis=1)]


In [11]:
plt.figure(figsize=(14,8))
plt.imshow(Q.T, 'bwr', aspect='auto', origin='lower', vmin=-10, vmax=10, interpolation='bicubic')
plt.tight_layout()


In [17]:
plt.figure(figsize=(10,8))
plt.plot(lat, Q[18])
plt.plot(lat, Q[23])

plt.ylim(-15,15)
plt.tight_layout()